# UFA Data Ingestion Pipeline

This notebook orchestrates the complete data pipeline:
1. Fetch metadata (teams, players, games) from UFA API
2. Clean and process game events
3. Insert everything into PostgreSQL database

**Prerequisites:**
- PostgreSQL running locally with `ufa_analytics` database
- Schema created (run `postgres_setup.ipynb` first)
- Functions defined in `data_cleaning_functions.ipynb`

## Setup and Configuration

In [51]:
# Import functions from other notebooks
import sys
sys.path.append('.')

# We'll import from the notebooks by running them
%run data_cleaning_functions.ipynb
%run postgres_setup.ipynb

Found 24 teams
Sample: {'team_id': 'aviators', 'team_name': 'LA Aviators', 'division': 'West', 'year': 2024}

Found 9 games in April 2024
Sample: {'game_id': '2024-04-27-MIN-PIT', 'home_team_id': 'thunderbirds', 'away_team_id': 'windchill', 'home_score': 11, 'away_score': 18, 'year': 2024, 'game_date': '2024-04-27', 'status': 'Final'}

Found 870 player-team records for 2024
Sample: {'player_id': 'mkiyoi', 'full_name': 'Michael Kiyoi', 'team_id': 'aviators', 'year': 2024, 'active': True, 'jersey_number': '1'}
Game 2024-04-27-MTL-NY: 750 events
First 5 events: [{'type': 1, 'line': ['cguay', 'swarrick', 'jbernat', 'rlalondel', 'tlalondel', 'jduquette', 'ctremblay'], 'time': 0, 'team': 'royal', 'year': 2024, 'gameID': '2024-04-27-MTL-NY'}, {'type': 2, 'line': ['lhaberfie', 'bjagt', 'skeegan', 'ochartock', 'srueschem', 'cweinberg', 'jwilliams'], 'time': 0, 'team': 'empire', 'year': 2024, 'gameID': '2024-04-27-MTL-NY'}, {'type': 7, 'puller': 'ctremblay', 'pullX': -16.47, 'pullY': 89.19, 'pul

In [52]:
from datetime import datetime
from tqdm import tqdm  # Progress bars
import time

### Configuration

Set your ingestion parameters here:

In [ ]:
# Configuration
CONFIG = {
    'year': '2026',                     # Year to ingest
    'date_range': '2026-01:2026-12',    # Date range for games (or use year)
    'game_status': 'Final',             # Only get completed games
    'batch_size': 10,                   # Process games in batches
    'skip_existing': True,              # Reprocess all games with new coordinate averaging!
}

print("Pipeline Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Pipeline Configuration:
  year: 2025
  date_range: 2025-01:2025-12
  game_status: Final
  batch_size: 10
  skip_existing: False


## Step 1: Fetch Teams and Players Metadata

In [54]:
print("Fetching teams data...")
teams = get_teams_data(years=CONFIG['year'])
print(f"✅ Fetched {len(teams)} teams")

# Display sample
if teams:
    print(f"\nSample team: {teams[0]}")

Fetching teams data...
✅ Fetched 26 teams

Sample team: {'team_id': 'phoenix', 'team_name': 'Philadelphia Phoenix', 'division': 'East', 'year': 2025}


In [55]:
print("Fetching players data...")
players = get_players_data(years=CONFIG['year'])
print(f"✅ Fetched {len(players)} player-team records")

# Display sample
if players:
    print(f"\nSample player: {players[0]}")

Fetching players data...
✅ Fetched 921 player-team records

Sample player: {'player_id': 'czaleski', 'full_name': 'Christopher Zaleski', 'team_id': 'phoenix', 'year': 2025, 'active': True, 'jersey_number': '0'}


### Insert Teams and Players into Database (Optional)

Note: The events insertion will work without pre-populating these tables,
but you can insert them here if you want to enforce foreign key constraints later.

In [56]:
# Insert teams and players into database
print("Inserting teams into database...")
insert_teams(teams)

print("\nInserting players into database...")
insert_players(players)

print("\n✅ Teams and players successfully inserted!")

Inserting teams into database...
✅ Inserted/updated 26 teams

Inserting players into database...
✅ Inserted/updated 921 players

✅ Teams and players successfully inserted!


## Step 2: Fetch Games List

In [57]:
print(f"Fetching games for {CONFIG['date_range']}...")
games = get_games(
    date=CONFIG['date_range'],
    statuses=CONFIG['game_status']
)

print(f"✅ Found {len(games)} games")

if games:
    print(f"\nDate range: {games[0]['game_date']} to {games[-1]['game_date']}")
    print(f"Sample game: {games[0]}")

Fetching games for 2025-01:2025-12...
✅ Found 157 games

Date range: 2025-04-24 to 2025-08-23
Sample game: {'game_id': '2025-04-24-NY-OAK', 'home_team_id': 'spiders', 'away_team_id': 'empire', 'home_score': 22, 'away_score': 17, 'year': 2025, 'game_date': '2025-04-24', 'status': 'Final'}


<cell_type>markdown</cell_type>## Step 3: Process and Insert Game Events

For each game:
1. Fetch game events from API
2. Clean and merge home/away events
3. Insert missing turnovers
4. **Normalize coordinates** (flip away team to home team's physical frame)
5. **Average turnover coordinates** (fixes team recording mismatches)
6. Insert into database

In [58]:
# Statistics tracking
stats = {
    'total_games': len(games),
    'processed': 0,
    'skipped': 0,
    'errors': 0,
    'total_events': 0,
    'synthetic_turnovers': 0
}

errors = []  # Track errors for review

In [59]:
print(f"\nProcessing {len(games)} games...\n")
print("=" * 80)

for i, game in enumerate(tqdm(games, desc="Processing games"), 1):
    game_id = game['game_id']
    
    try:
        # Check if game already exists (if skip_existing is enabled)
        if CONFIG['skip_existing']:
            conn = get_connection()
            cur = conn.cursor()
            cur.execute("SELECT COUNT(*) FROM games WHERE game_id = %s", (game_id,))
            exists = cur.fetchone()[0] > 0
            cur.close()
            conn.close()
            
            if exists:
                stats['skipped'] += 1
                continue
        
        # Fetch and clean events
        cleaned_data = clean_data([game_id])
        
        if not cleaned_data or game_id not in cleaned_data:
            print(f"\n⚠️ No data for game {game_id}")
            stats['errors'] += 1
            errors.append({'game_id': game_id, 'error': 'No event data returned'})
            continue
        
        events = cleaned_data[game_id]
        
        # Insert missing turnovers
        fixed_events = insert_missing_turnovers(events, seed=42)
        synthetic_count = len(fixed_events) - len(events)

        # Normalize coordinates (flip away team to home team's physical frame)
        normalized_events = normalize_coordinates(fixed_events, game['away_team_id'])

        # Average turnover coordinates (both teams now in same frame)
        averaged_events = average_turnover_coordinates(normalized_events)
        
        # Insert into database
        insert_game_data(
            game_id=game_id,
            events=averaged_events,
            home_team_id=game['home_team_id'],
            away_team_id=game['away_team_id'],
            home_score=game['home_score'],
            away_score=game['away_score'],
            year=game['year']
        )
        
        # Update stats
        stats['processed'] += 1
        stats['total_events'] += len(averaged_events)
        stats['synthetic_turnovers'] += synthetic_count
        
        # Progress update every 10 games
        if i % 10 == 0:
            print(f"\n✅ Processed {i}/{len(games)} games")
        
        # Brief pause to avoid overwhelming the API
        time.sleep(0.1)
        
    except Exception as e:
        print(f"\n❌ Error processing game {game_id}: {e}")
        stats['errors'] += 1
        errors.append({'game_id': game_id, 'error': str(e)})
        continue

print("\n" + "=" * 80)
print("Processing complete!")
print("=" * 80)


Processing 157 games...



Processing games:   0%|          | 0/157 [00:00<?, ?it/s]


Inserting data for game: 2025-04-24-NY-OAK
✅ Inserted game: 2025-04-24-NY-OAK
✅ Inserted 708 events for game 2025-04-24-NY-OAK
✅ Inserted 672 line player records for game 2025-04-24-NY-OAK

✅ Complete! Game 2025-04-24-NY-OAK inserted successfully.



Processing games:   1%|          | 1/157 [00:00<01:51,  1.39it/s]


Inserting data for game: 2025-06-07-PHI-BOS
✅ Inserted game: 2025-06-07-PHI-BOS
✅ Inserted 700 events for game 2025-06-07-PHI-BOS
✅ Inserted 525 line player records for game 2025-06-07-PHI-BOS

✅ Complete! Game 2025-06-07-PHI-BOS inserted successfully.



...


Inserting data for game: 2025-07-05-CHI-PIT
✅ Inserted game: 2025-07-05-CHI-PIT
✅ Inserted 702 events for game 2025-07-05-CHI-PIT
✅ Inserted 665 line player records for game 2025-07-05-CHI-PIT

✅ Complete! Game 2025-07-05-CHI-PIT inserted successfully.



Processing games:  99%|█████████▉| 156/157 [01:54<00:00,  1.37it/s]


Inserting data for game: 2025-08-23-BOS-MIN
✅ Inserted game: 2025-08-23-BOS-MIN
✅ Inserted 760 events for game 2025-08-23-BOS-MIN
✅ Inserted 581 line player records for game 2025-08-23-BOS-MIN

✅ Complete! Game 2025-08-23-BOS-MIN inserted successfully.



Processing games: 100%|██████████| 157/157 [01:55<00:00,  1.36it/s]


Processing complete!


## Pipeline Summary

In [60]:
print("\n" + "=" * 80)
print("PIPELINE SUMMARY")
print("=" * 80)
print(f"\nTotal games found: {stats['total_games']}")
print(f"Successfully processed: {stats['processed']}")
print(f"Skipped (already in DB): {stats['skipped']}")
print(f"Errors: {stats['errors']}")
print(f"\nTotal events inserted: {stats['total_events']:,}")
print(f"Synthetic turnovers added: {stats['synthetic_turnovers']}")

if stats['processed'] > 0:
    avg_events = stats['total_events'] / stats['processed']
    print(f"Average events per game: {avg_events:.1f}")

if errors:
    print(f"\n⚠️ {len(errors)} errors occurred:")
    for error in errors[:5]:  # Show first 5 errors
        print(f"  - {error['game_id']}: {error['error']}")
    if len(errors) > 5:
        print(f"  ... and {len(errors) - 5} more")

print("\n" + "=" * 80)


PIPELINE SUMMARY

Total games found: 157
Successfully processed: 157
Skipped (already in DB): 0
Errors: 0

Total events inserted: 125,165
Synthetic turnovers added: 37
Average events per game: 797.2



## Verification Queries

In [61]:
# Verify data was inserted correctly
conn = get_connection()
cur = conn.cursor()

# Count total games
cur.execute("SELECT COUNT(*) FROM games;")
total_games = cur.fetchone()[0]
print(f"Total games in database: {total_games}")

# Count total events
cur.execute("SELECT COUNT(*) FROM events;")
total_events = cur.fetchone()[0]
print(f"Total events in database: {total_events:,}")

# Count synthetic events
cur.execute("SELECT COUNT(*) FROM events WHERE synthetic = TRUE;")
synthetic_events = cur.fetchone()[0]
print(f"Synthetic events: {synthetic_events}")

# Count line player records
cur.execute("SELECT COUNT(*) FROM line_players;")
line_players_count = cur.fetchone()[0]
print(f"Line player records: {line_players_count:,}")

# Events by type
cur.execute("""
    SELECT event_type, COUNT(*) as count
    FROM events
    GROUP BY event_type
    ORDER BY event_type;
""")

print("\nEvents by type:")
for row in cur.fetchall():
    print(f"  Type {row[0]}: {row[1]:,} events")

cur.close()
conn.close()

Total games in database: 762
Total events in database: 560,616
Synthetic events: 158
Line player records: 505,931

Events by type:
  Type 1: 32,918 events
  Type 2: 32,904 events
  Type 3: 3,815 events
  Type 7: 30,354 events
  Type 8: 2,288 events
  Type 11: 14,669 events
  Type 16: 11,134 events
  Type 17: 11,135 events
  Type 18: 353,733 events
  Type 19: 30,165 events
  Type 20: 4,213 events
  Type 22: 24,006 events
  Type 23: 116 events
  Type 24: 414 events
  Type 25: 2,690 events
  Type 28: 1,514 events
  Type 29: 1,499 events
  Type 30: 1,504 events
  Type 31: 1,441 events
  Type 32: 80 events
  Type 33: 24 events


## Done! 🎉

Your UFA data has been successfully ingested into the PostgreSQL database.

### Next Steps:
- Query the database for analysis
- Build visualizations
- Create player/team statistics
- Analyze line combinations and synergies